# Detect object with YOLOv5 and OpenCV

## Sources:
- [Object Detection using YOLOv5 and OpenCV DNN in C++ and Python](https://learnopencv.com/object-detection-using-yolov5-and-opencv-dnn-in-c-and-python/?ck_subscriber_id=1558025914)

## Import modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
from datetime import date


# import 3rd-party modules
import cv2
import numpy as np


# import local modules
from utils.renderer.giffer import create_gif
from utils.renderer.videographer import create_video
from utils.project_manager import Project
from utils.renderer.resizer import get_interpolation
from utils.renderer.resizer import resize_with_pad, resize_with_crop

## Define Global Parameters

In [ ]:
# set blob size (blob: binary large object; contains the data in readable raw format;
# image has to be converted to a blob so as the network can process it)
INPUT_HEIGHT, INPUT_WIDTH = 640, 640

# set low probability class filter
SCORE_THRESHOLD = 0.5

# set overlapping bounding boxes filter
NMS_THRESHOLD = 0.45

# set low probability detection filter
CONFIDENCE_THRESHOLD = 0.45

# set text parameters
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.7
THICKNESS = 1

# set colors
BLACK = (0,0,0)
BLUE = (255, 178, 50)
YELLOW = (0,255, 255)

## Define functions

In [ ]:
def draw_label(img, label, x, y):
    """
    Function to draw text onto image at xy coords
    """
    # get text size
    text_size = cv2.getTextSize(label, FONT, FONT_SCALE, THICKNESS)
    dim, baseline = text_size[0], text_size[1]

    # use text size to create a black rectangle
    cv2.rectangle(img, (x,y), (x + dim[0], y + dim[1] + baseline), BLACK, cv2.FILLED)

    # display text inside rectangle
    cv2.putText(img, label, (x, y + dim[1]), FONT, FONT_SCALE, YELLOW, THICKNESS, cv2.LINE_AA)

def pre_process(input_img, net):
    """
    Function to 
    - convert image to a blob of a 4D array object
    - pass it to the neural network

    Returns a 2D array of shape (25200, 85) or from OpenCV-Python 4.5.5, a 3D array of shape (1, 25200, 85)
    - rows = number of detections (=> number of bounding boxes)
    - columns = 85 info of each detection: x, y, width, height, confidence, class scores of 80 classes
        * x,y : normalized center coords of detected bounding box
        * width, height: normalized width and height
        * confidence: probability of detection being an object
        * class scores of 80 objects from COCO dataset 2017 (on which model has been trained)
    """
    # create a 4d blob from a frame
    blob = cv2.dnn.blobFromImage(input_img, 1/255, (INPUT_WIDTH, INPUT_HEIGHT), [0,0,0], 1, crop=False)

    # set input to the network
    net.setInput(blob)

    # run the forward pass to get output of the output layers
    outputs = net.forward(net.getUnconnectedOutLayersNames())

    return outputs

def post_process(input_img, outputs):
    """
    Function to unwrap the outputs from the neural network
    """
    # create empty lists to store 
    # Lists to hold respective values while unwrapping.
    class_ids = []
    confidences = []
    boxes = []
    # Rows.
    rows = outputs[0].shape[1]
    image_height, image_width = input_img.shape[:2]
    # Resizing factor.
    x_factor = image_width / INPUT_WIDTH
    y_factor =  image_height / INPUT_HEIGHT
    # Iterate through 25200 detections.
    for r in range(rows):
        row = outputs[0][0][r]
        confidence = row[4]
        # Discard bad detections and continue.
        if confidence >= CONFIDENCE_THRESHOLD:
                classes_scores = row[5:]
                # Get the index of max class score.
                class_id = np.argmax(classes_scores)
                #  Continue if the class score is above threshold.
                if (classes_scores[class_id] > SCORE_THRESHOLD):
                    confidences.append(confidence)
                    class_ids.append(class_id)
                    cx, cy, w, h = row[0], row[1], row[2], row[3]
                    left = int((cx - w/2) * x_factor)
                    top = int((cy - h/2) * y_factor)
                    width = int(w * x_factor)
                    height = int(h * y_factor)
                    box = np.array([left, top, width, height])
                    boxes.append(box)

Help on built-in function blobFromImage:

blobFromImage(...)
    blobFromImage(image[, scalefactor[, size[, mean[, swapRB[, crop[, ddepth]]]]]]) -> retval
    .   @brief Creates 4-dimensional blob from image. Optionally resizes and crops @p image from center,
    .        *  subtract @p mean values, scales values by @p scalefactor, swap Blue and Red channels.
    .        *  @param image input image (with 1-, 3- or 4-channels).
    .        *  @param size spatial size for output image
    .        *  @param mean scalar with mean values which are subtracted from channels. Values are intended
    .        *  to be in (mean-R, mean-G, mean-B) order if @p image has BGR ordering and @p swapRB is true.
    .        *  @param scalefactor multiplier for @p image values.
    .        *  @param swapRB flag which indicates that swap first and last channels
    .        *  in 3-channel image is necessary.
    .        *  @param crop flag which indicates whether image will be cropped after resize o